# 03_split.ipynb - Train/Validation/Test Split

**Project**: BSM Project 3 - D-ResNet Paper Reproduction
**Step**: 3 of 9 - Data Splitting

## Objectives
1. Load combined dataset (original + augmented) from Step 2
2. **Primary split**: 70/15/15 stratified on combined set (paper-faithful)
   - ~1028 train / 220 val / 220 test (from 1468 total)
3. **Supplementary split**: Leak-free - split originals first, then augment only train
   - Split 367 originals 70/15/15, then augment training set only
4. Save split file lists to `data_info/` for reproducibility across all model notebooks
5. Fixed random seed (42) everywhere

In [ ]:
# ========== CONFIGURATION ==========
SEED = 42
COMBINED_DATA_DIR = "/kaggle/working/combined_dataset"  # From Step 2
# Alternative local path:
# COMBINED_DATA_DIR = "C:/BSM Project 3/data_raw/BSM"  # For testing without augmentation

ORIGINAL_DATA_DIR = "/kaggle/input/beach-sand-mineral-bsm/BSM"  # Original dataset
# Alternative local path:
# ORIGINAL_DATA_DIR = "C:/BSM Project 3/data_raw/BSM"

EXPECTED_CLASSES = ["Garnet", "Ilmenite", "Monazite", "Rutile", "Sillimanite", "Zircon"]

# Split ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Output paths
DATA_INFO_DIR = "/kaggle/working/data_info"
RESULTS_DIR = "/kaggle/working/results"

import os
os.makedirs(DATA_INFO_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ========== IMPORTS & SEED ==========
import os
import sys
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
import shutil
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

# Set seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)

print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Seed: {SEED}")

In [ ]:
# ========== LOAD COMBINED DATASET (Primary Split) ==========
combined_path = Path(COMBINED_DATA_DIR)
print(f"Combined data path: {combined_path}")
print(f"Exists: {combined_path.exists()}")

combined_images = []
combined_class_counts = {}

for class_dir in sorted(combined_path.iterdir()):
    if not class_dir.is_dir():
        continue
    class_name = class_dir.name
    
    img_files = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.jpeg")) + \
               list(class_dir.glob("*.png"))
    
    combined_class_counts[class_name] = len(img_files)
    for img_path in img_files:
        combined_images.append((str(img_path), class_name))
    
    print(f"  {class_name}: {len(img_files)} images")

total_combined = len(combined_images)
print(f"\nTotal combined images: {total_combined}")

In [ ]:
# ========== PRIMARY SPLIT: 70/15/15 ON COMBINED SET ==========
# This matches the paper's approach (split after augmentation)
# WARNING: This can leak augmented copies of same original across splits

paths = [p for p, _ in combined_images]
labels = [c for _, c in combined_images]

# First split: 70% train, 30% temp (val + test)
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    paths, labels,
    train_size=TRAIN_RATIO,
    stratify=labels,
    random_state=SEED
)

# Second split: 50/50 of temp = 15% val, 15% test
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels,
    train_size=0.5,
    stratify=temp_labels,
    random_state=SEED
)

print(f"Primary Split (Combined):")
print(f"  Train: {len(train_paths)} ({len(train_paths)/total_combined*100:.1f}%)")
print(f"  Val:   {len(val_paths)} ({len(val_paths)/total_combined*100:.1f}%)")
print(f"  Test:  {len(test_paths)} ({len(test_paths)/total_combined*100:.1f}%)")

# Verify stratification
for split_name, split_labels in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    dist = Counter(split_labels)
    print(f"  {split_name} distribution: {dict(sorted(dist.items()))}")

In [ ]:
# ========== LOAD ORIGINAL DATASET (Supplementary Split) ==========
original_path = Path(ORIGINAL_DATA_DIR)
print(f"Original data path: {original_path}")
print(f"Exists: {original_path.exists()}")

original_images = []
original_class_counts = {}

for class_dir in sorted(original_path.iterdir()):
    if not class_dir.is_dir():
        continue
    class_name = class_dir.name
    
    img_files = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.jpeg")) + \
               list(class_dir.glob("*.png")) + list(class_dir.glob("*.bmp")) + \
               list(class_dir.glob("*.tif")) + list(class_dir.glob("*.tiff"))
    
    valid_images = []
    for img_path in img_files:
        try:
            with Image.open(img_path) as img:
                img.verify()
            valid_images.append(str(img_path))
        except:
            pass
    
    original_class_counts[class_name] = len(valid_images)
    for img_path in valid_images:
        original_images.append((img_path, class_name))
    
    print(f"  {class_name}: {len(valid_images)} images")

total_original = len(original_images)
print(f"\nTotal original images: {total_original}")

In [ ]:
# ========== SUPPLEMENTARY SPLIT: LEAK-FREE ==========
# Split originals first (70/15/15), then augment ONLY training set
# This prevents data leakage

orig_paths = [p for p, _ in original_images]
orig_labels = [c for _, c in original_images]

# Split originals
train_orig_paths, temp_orig_paths, train_orig_labels, temp_orig_labels = train_test_split(
    orig_paths, orig_labels,
    train_size=TRAIN_RATIO,
    stratify=orig_labels,
    random_state=SEED
)

val_orig_paths, test_orig_paths, val_orig_labels, test_orig_labels = train_test_split(
    temp_orig_paths, temp_orig_labels,
    train_size=0.5,
    stratify=temp_orig_labels,
    random_state=SEED
)

print(f"Supplementary Split (Originals only):")
print(f"  Train: {len(train_orig_paths)} ({len(train_orig_paths)/total_original*100:.1f}%)")
print(f"  Val:   {len(val_orig_paths)} ({len(val_orig_paths)/total_original*100:.1f}%)")
print(f"  Test:  {len(test_orig_paths)} ({len(test_orig_paths)/total_original*100:.1f}%)")

# Verify stratification
for split_name, split_labels in [("Train", train_orig_labels), ("Val", val_orig_labels), ("Test", test_orig_labels)]:
    dist = Counter(split_labels)
    print(f"  {split_name} distribution: {dict(sorted(dist.items()))}")

In [ ]:
# ========== SAVE PRIMARY SPLIT LISTS ==========
# Save as JSON and CSV for easy loading in model notebooks

primary_split = {
    "split_type": "primary_combined",
    "description": "70/15/15 stratified split on combined (original + augmented) dataset. Paper-faithful but may leak augmented copies.",
    "seed": SEED,
    "total_images": total_combined,
    "train": [{"path": p, "class": c} for p, c in zip(train_paths, train_labels)],
    "val": [{"path": p, "class": c} for p, c in zip(val_paths, val_labels)],
    "test": [{"path": p, "class": c} for p, c in zip(test_paths, test_labels)]
}

with open(os.path.join(DATA_INFO_DIR, 'primary_split.json'), 'w') as f:
    json.dump(primary_split, f, indent=2)

# CSV format for easy reading
train_df = pd.DataFrame([{"path": p, "class": c, "split": "train"} for p, c in zip(train_paths, train_labels)])
val_df = pd.DataFrame([{"path": p, "class": c, "split": "val"} for p, c in zip(val_paths, val_labels)])
test_df = pd.DataFrame([{"path": p, "class": c, "split": "test"} for p, c in zip(test_paths, test_labels)])

primary_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
primary_df.to_csv(os.path.join(DATA_INFO_DIR, 'primary_split.csv'), index=False)

print(f"Primary split saved to {DATA_INFO_DIR}/primary_split.json and .csv")

In [ ]:
# ========== SAVE SUPPLEMENTARY SPLIT LISTS ==========
# For leak-free evaluation, we save the original-image-level splits
# Model notebooks will need to augment training set only for this split

supplementary_split = {
    "split_type": "supplementary_leak_free",
    "description": "70/15/15 split on original 367 images only. Training set augmented separately. No leakage.",
    "seed": SEED,
    "total_originals": total_original,
    "train_originals": [{"path": p, "class": c} for p, c in zip(train_orig_paths, train_orig_labels)],
    "val_originals": [{"path": p, "class": c} for p, c in zip(val_orig_paths, val_orig_labels)],
    "test_originals": [{"path": p, "class": c} for p, c in zip(test_orig_paths, test_orig_labels)]
}

with open(os.path.join(DATA_INFO_DIR, 'supplementary_split.json'), 'w') as f:
    json.dump(supplementary_split, f, indent=2)

# CSV format
train_orig_df = pd.DataFrame([{"path": p, "class": c, "split": "train"} for p, c in zip(train_orig_paths, train_orig_labels)])
val_orig_df = pd.DataFrame([{"path": p, "class": c, "split": "val"} for p, c in zip(val_orig_paths, val_orig_labels)])
test_orig_df = pd.DataFrame([{"path": p, "class": c, "split": "test"} for p, c in zip(test_orig_paths, test_orig_labels)])

supplementary_df = pd.concat([train_orig_df, val_orig_df, test_orig_df], ignore_index=True)
supplementary_df.to_csv(os.path.join(DATA_INFO_DIR, 'supplementary_split.csv'), index=False)

print(f"Supplementary split saved to {DATA_INFO_DIR}/supplementary_split.json and .csv")

In [ ]:
# ========== VISUALIZE SPLIT DISTRIBUTIONS ==========
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Primary split distributions
splits = [
    ("Primary Train", train_labels),
    ("Primary Val", val_labels),
    ("Primary Test", test_labels),
    ("Supp Train (Orig)", train_orig_labels),
    ("Supp Val (Orig)", val_orig_labels),
    ("Supp Test (Orig)", test_orig_labels),
]

for i, (name, labels) in enumerate(splits):
    dist = Counter(labels)
    classes = sorted(dist.keys())
    counts = [dist[c] for c in classes]
    
    axes[i].bar(classes, counts, color=plt.cm.Set2(np.linspace(0, 1, len(classes))), edgecolor='black')
    axes[i].set_title(name)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45)
    
    for j, count in enumerate(counts):
        axes[i].text(j, count + 0.5, str(count), ha='center', va='bottom', fontsize=8)

plt.suptitle('Split Distributions: Primary (Combined) vs Supplementary (Leak-Free)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'split_distributions.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ========== CHECK FOR LEAKAGE IN PRIMARY SPLIT ==========
# Check if augmented copies of same original appear in different splits

def get_original_stem(filename):
    """Extract original image stem from augmented filename."""
    # Format: original_stem_augTRANSFORM.jpg
    if '_aug' in filename:
        return filename.split('_aug')[0]
    return Path(filename).stem

# Get original stems for each split
train_stems = set(get_original_stem(Path(p).name) for p in train_paths)
val_stems = set(get_original_stem(Path(p).name) for p in val_paths)
test_stems = set(get_original_stem(Path(p).name) for p in test_paths)

# Check overlaps
train_val_overlap = train_stems & val_stems
train_test_overlap = train_stems & test_stems
val_test_overlap = val_stems & test_stems

print(f"Leakage Analysis (Primary Split):")
print(f"  Train-Val overlap: {len(train_val_overlap)} original images")
print(f"  Train-Test overlap: {len(train_test_overlap)} original images")
print(f"  Val-Test overlap: {len(val_test_overlap)} original images")

if train_test_overlap:
    print(f"  WARNING: {len(train_test_overlap)} originals have augmented copies in BOTH train and test!")
    for stem in sorted(train_test_overlap)[:10]:
        print(f"    - {stem}")
    if len(train_test_overlap) > 10:
        print(f"    ... and {len(train_test_overlap) - 10} more")

# Save leakage report
leakage_report = {
    "train_val_overlap_count": len(train_val_overlap),
    "train_test_overlap_count": len(train_test_overlap),
    "val_test_overlap_count": len(val_test_overlap),
    "train_val_overlap": sorted(list(train_val_overlap)),
    "train_test_overlap": sorted(list(train_test_overlap)),
    "val_test_overlap": sorted(list(val_test_overlap))
}

with open(os.path.join(DATA_INFO_DIR, 'leakage_report.json'), 'w') as f:
    json.dump(leakage_report, f, indent=2)

print(f"\nLeakage report saved to {DATA_INFO_DIR}/leakage_report.json")

In [ ]:
# ========== SUMMARY ==========
print("="*60)
print("SPLIT SUMMARY")
print("="*60)

print(f"\nPRIMARY SPLIT (Paper-faithful, combined dataset):")
print(f"  Total: {total_combined}")
print(f"  Train: {len(train_paths)} ({len(train_paths)/total_combined*100:.1f}%)")
print(f"  Val:   {len(val_paths)} ({len(val_paths)/total_combined*100:.1f}%)")
print(f"  Test:  {len(test_paths)} ({len(test_paths)/total_combined*100:.1f}%)")
print(f"  Leakage (train-test): {len(train_test_overlap)} originals")

print(f"\nSUPPLEMENTARY SPLIT (Leak-free, originals only):")
print(f"  Total originals: {total_original}")
print(f"  Train: {len(train_orig_paths)} ({len(train_orig_paths)/total_original*100:.1f}%)")
print(f"  Val:   {len(val_orig_paths)} ({len(val_orig_paths)/total_original*100:.1f}%)")
print(f"  Test:  {len(test_orig_paths)} ({len(test_orig_paths)/total_original*100:.1f}%)")
print(f"  (Training set to be augmented separately in model notebooks)")

print(f"\nFiles saved to {DATA_INFO_DIR}/:")
print(f"  - primary_split.json / .csv")
print(f"  - supplementary_split.json / .csv")
print(f"  - leakage_report.json")
print(f"  - split_distributions.png (in results/)")